# HCDE 530: Tidy Data — In-Class Exercise
## Cleaning UX Research Data from the FlowNote Study

In this exercise you'll work with three small datasets collected during a UX research
study of **FlowNote**, an imaginary note-taking app for design teams. Each dataset has
a different structural problem that makes it hard to analyze.

Your job for each one:
1. **Explore** the data and spot the tidy violation
2. **Describe** the problem precisely
3. **Prompt** your AI agent to write pandas code that fixes it
4. **Verify** the result using the provided check cell

**By the end of this session you should be able to:**
- Identify at least three common ways tabular data can violate tidy principles
- Write precise prompts that describe a data-tidying problem to an AI agent
- Use `pd.melt()`, `str.split()`, and DataFrame subsetting to restructure data
- Evaluate whether a result satisfies Wickham's three tidy rules

---
## Quick Reference: What Makes Data Tidy?

Wickham (2014) defines a tidy dataset as one satisfying all three rules:

| Rule | Description |
|------|-------------|
| **1. Each variable is a column** | Every distinct attribute you measure gets its own column |
| **2. Each observation is a row** | Every instance of the thing you are measuring gets its own row |
| **3. Each table holds one type of unit** | Don't mix data about users with data about products in the same table |

**The five most common violations:**
1. Column headers are values, not variable names
2. Multiple variables packed into one column
3. Variables stored in both rows and columns
4. Multiple observational units in one table
5. One observational unit spread across multiple tables

Keep these in mind as you work through the exercises.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("pandas version:", pd.__version__)

---
## Exercise 1: Weekly Satisfaction Ratings
### Violation: Column Headers Are Values

The FlowNote team ran a **five-week onboarding study**. At the end of each week,
participants rated their overall satisfaction with the app on a 1–5 scale.
The data analyst exported results in the format below.

In [ ]:
data1 = {
    'user_id':  ['U001','U002','U003','U004','U005',
                 'U006','U007','U008','U009','U010'],
    'name':     ['Alex Chen','Priya Patel','Jordan Kim','Sam Okafor','Taylor Reyes',
                 'Morgan Liu','Casey Brown','Riley Zhang','Drew Santos','Jamie Nguyen'],
    'role':     ['UX Designer','Product Manager','Developer','UX Researcher',
                 'Content Strategist','UX Designer','Product Manager',
                 'Developer','UX Researcher','UX Designer'],
    'week_1':   [3.8, 4.2, 3.5, 4.0, 3.9, 4.1, 3.7, 3.6, 4.3, 3.8],
    'week_2':   [4.0, 4.3, 3.8, 4.2, 4.0, 4.2, 3.9, 3.8, 4.4, 4.0],
    'week_3':   [4.2, 4.4, 4.1, 4.3, 4.2, 4.4, 4.2, 4.0, 4.5, 4.1],
    'week_4':   [4.3, 4.5, 4.3, 4.4, 4.3, 4.5, 4.4, 4.2, 4.6, 4.3],
    'week_5':   [4.5, 4.7, 4.5, 4.6, 4.5, 4.6, 4.5, 4.4, 4.8, 4.5],
}

df1 = pd.DataFrame(data1)
df1

### Step 1 — Explore the data

Run the cell below and study the output carefully.

- How many rows and columns are there?
- What does each **row** represent?
- What does each **column** represent?

In [ ]:
print("Shape:", df1.shape)
print()
print("Column names:", df1.columns.tolist())
print()
print("Data types:")
print(df1.dtypes)

### Step 2 — Spot the violation

Look at the columns `week_1` through `week_5`.

> **Think about it:** What *variable* does the column name `week_1` represent?
> Is the week number truly a variable *name* — or is it a *value* of some
> underlying variable called "week"?

According to tidy rule #1, each variable should form a column. Here, **week** is
a variable and **satisfaction score** is another — but both are collapsed into the
column names. The five `week_N` columns are actually values of a `week` variable,
not separate variables themselves.

**The fix:** Reshape from **wide format** to **long format**. The `week_N` columns
need to be "melted" into two new columns: one for the week identifier and one for
the score. The tidy result should have **50 rows** (10 users × 5 weeks), not 10.

The pandas function for this is `pd.melt()`.

---
### Step 3 — Prompt your AI agent

Before writing any code, write a prompt for your AI agent. A good prompt for a
data-tidying task includes:

1. **Current structure** — what the DataFrame looks like now
2. **The problem** — why it violates tidy principles
3. **The goal** — what the tidy version should look like, including column names

**Weak prompt** (don't use this):
> *"Fix my data."*

**Stronger prompt** (model yours on this):
> *"I have a pandas DataFrame called `df1` with columns: `user_id`, `name`, `role`,
> `week_1`, `week_2`, `week_3`, `week_4`, `week_5`. The `week_N` columns contain
> satisfaction scores, but 'week' should be a variable, not a column name. Tidy
> the data using `pd.melt()` so that the result has one row per user-week
> combination. The melted columns should become two new columns called `week`
> (values like 'week_1') and `satisfaction_score`. Keep `user_id`, `name`, and
> `role` as identifier columns. Assign the result to `df1_tidy` and display it."*

Notice how the strong prompt names the DataFrame, lists the columns, explains the
tidy problem, specifies the output column names, and states what to assign the
result to.

**Write your prompt in the cell below, then try it with your agent.**

### My Prompt (Exercise 1)

*Write your prompt here before moving to the code cell.*

```
[your prompt here]
```

In [ ]:
# Paste the code your AI agent produced here, then run it.
# The result should be assigned to df1_tidy.

# ── YOUR CODE HERE ──────────────────────────────────────────────────────────

# df1_tidy = ...

In [ ]:
# ── Verification — run this cell to check your work ─────────────────────────

try:
    assert 'week' in df1_tidy.columns, "Missing column: 'week'"
    assert 'satisfaction_score' in df1_tidy.columns, "Missing column: 'satisfaction_score'"
    assert len(df1_tidy) == 50, f"Expected 50 rows, got {len(df1_tidy)}"
    assert df1_tidy['week'].nunique() == 5, "Expected 5 unique week values"
    assert df1_tidy['user_id'].nunique() == 10, "Expected 10 unique users"
    print("✅  Tidy check passed!")
    print(f"   Shape: {df1_tidy.shape}")
    print(f"   Columns: {df1_tidy.columns.tolist()}")
    print()
    print(df1_tidy.sort_values(['user_id', 'week']).head(10).to_string(index=False))
except NameError:
    print("⚠️   df1_tidy is not defined — paste your code in the cell above and run it.")
except AssertionError as e:
    print(f"❌  Check failed: {e}")

---
## Exercise 2: Usability Test Session Log
### Violation: Multiple Variables Stored in One Column

After the onboarding study, the team ran a structured usability test with eight
participants across five tasks. The session-logger script recorded participant and
outcome information — but whoever designed the export packed multiple pieces of
information into single columns.

In [ ]:
data2 = {
    'session_id': [f'S{i:03d}' for i in range(1, 41)],
    'participant_info': [
        # 8 participants × 5 tasks = 40 sessions
        'ux_designer/expert',           'ux_designer/expert',
        'ux_designer/expert',           'ux_designer/expert',
        'ux_designer/expert',
        'product_manager/intermediate', 'product_manager/intermediate',
        'product_manager/intermediate', 'product_manager/intermediate',
        'product_manager/intermediate',
        'developer/expert',             'developer/expert',
        'developer/expert',             'developer/expert',
        'developer/expert',
        'ux_researcher/intermediate',   'ux_researcher/intermediate',
        'ux_researcher/intermediate',   'ux_researcher/intermediate',
        'ux_researcher/intermediate',
        'content_writer/novice',        'content_writer/novice',
        'content_writer/novice',        'content_writer/novice',
        'content_writer/novice',
        'ux_designer/intermediate',     'ux_designer/intermediate',
        'ux_designer/intermediate',     'ux_designer/intermediate',
        'ux_designer/intermediate',
        'product_manager/novice',       'product_manager/novice',
        'product_manager/novice',       'product_manager/novice',
        'product_manager/novice',
        'developer/intermediate',       'developer/intermediate',
        'developer/intermediate',       'developer/intermediate',
        'developer/intermediate',
    ],
    'task': [
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
        'create_note','search_content','share_note','set_reminder','organize_tags',
    ],
    'date': [
        '2025-03-10','2025-03-10','2025-03-10','2025-03-10','2025-03-10',
        '2025-03-11','2025-03-11','2025-03-11','2025-03-11','2025-03-11',
        '2025-03-12','2025-03-12','2025-03-12','2025-03-12','2025-03-12',
        '2025-03-13','2025-03-13','2025-03-13','2025-03-13','2025-03-13',
        '2025-03-14','2025-03-14','2025-03-14','2025-03-14','2025-03-14',
        '2025-03-17','2025-03-17','2025-03-17','2025-03-17','2025-03-17',
        '2025-03-18','2025-03-18','2025-03-18','2025-03-18','2025-03-18',
        '2025-03-19','2025-03-19','2025-03-19','2025-03-19','2025-03-19',
    ],
    'session_outcome': [
        # format: "status-duration_in_seconds"
        'completed-42',  'completed-31',  'completed-18',  'completed-27',  'completed-35',
        'completed-89',  'completed-76',  'failed-120',    'completed-54',  'completed-67',
        'completed-38',  'completed-29',  'completed-22',  'completed-31',  'completed-44',
        'completed-71',  'completed-65',  'completed-44',  'failed-115',    'completed-58',
        'failed-150',    'completed-112', 'failed-180',    'completed-95',  'failed-162',
        'completed-58',  'completed-49',  'completed-33',  'completed-41',  'completed-52',
        'failed-143',    'completed-98',  'failed-167',    'failed-134',    'completed-88',
        'completed-61',  'completed-47',  'completed-38',  'completed-52',  'completed-55',
    ],
    'confidence_score': [
        5, 5, 4, 5, 4,
        3, 4, 2, 3, 4,
        5, 5, 5, 4, 5,
        4, 3, 4, 2, 3,
        2, 3, 1, 3, 2,
        4, 4, 4, 3, 4,
        2, 3, 2, 2, 3,
        4, 4, 5, 4, 4,
    ],
}

df2 = pd.DataFrame(data2)
df2.head(12)

### Step 1 — Explore the data

Study the `participant_info` and `session_outcome` columns closely.

In [ ]:
print("Shape:", df2.shape)
print()
print("Unique participant_info values:")
for v in sorted(df2['participant_info'].unique()):
    print(" ", v)
print()
print("Sample session_outcome values:")
print(df2['session_outcome'].head(10).tolist())

### Step 2 — Spot the violations

> **Think about it:**
> - How many distinct pieces of information are stored in `participant_info`?
>   What separator is used?
> - How many distinct pieces of information are in `session_outcome`?
>   What separator is used there?

**The violations:**
- `participant_info` combines **role** and **skill_level** in one string (`/` separator)
- `session_outcome` combines **task_status** (completed/failed) and **duration_seconds**
  in one string (`-` separator)

Both violate tidy rule #1. Each of these pieces of information is a separate
variable and deserves its own column.

**The tidy result** should have these columns:

| session_id | role | skill_level | task | date | task_status | duration_seconds | confidence_score |
|---|---|---|---|---|---|---|---|
| S001 | ux_designer | expert | create_note | 2025-03-10 | completed | 42 | 5 |

Useful pandas string methods: `str.split()`, `str.get()`, `str.extract()`.
Your AI agent will know how to use them — your job is to describe the problem well.

---
### Step 3 — Prompt your AI agent

This time there are **two** combined columns to fix. Think about whether to handle
them in one prompt or two.

**Tip: include example values in your prompt.** When you give Claude a concrete
sample of what the data looks like, the output is almost always better than when
you describe it abstractly.

Here's a model prompt structure:
> *"I have a pandas DataFrame `df2` with a column `participant_info` that stores
> values like `'ux_designer/expert'` and `'product_manager/intermediate'`. The `/`
> separates role from skill level. Split it into two columns: `role` and
> `skill_level`. Also split `session_outcome` (values like `'completed-42'` and
> `'failed-120'`) into `task_status` and `duration_seconds`, where
> `duration_seconds` should be an integer. Drop the original combined columns.
> Assign the result to `df2_tidy`."*

**Write your own prompt below.**

### My Prompt (Exercise 2)

```
[your prompt here]
```

In [ ]:
# Paste the code your AI agent produced here.
# The result should be assigned to df2_tidy.

# ── YOUR CODE HERE ──────────────────────────────────────────────────────────

# df2_tidy = ...

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────

required = {'session_id', 'role', 'skill_level', 'task', 'date',
            'task_status', 'duration_seconds', 'confidence_score'}

try:
    missing = required - set(df2_tidy.columns)
    assert not missing, f"Missing columns: {missing}"
    assert len(df2_tidy) == 40, f"Expected 40 rows, got {len(df2_tidy)}"
    assert df2_tidy['skill_level'].nunique() == 3, (
        f"Expected 3 skill levels (novice/intermediate/expert), "
        f"got {df2_tidy['skill_level'].unique()}"
    )
    assert set(df2_tidy['task_status'].unique()) == {'completed', 'failed'}, (
        f"task_status should be 'completed' or 'failed', got {df2_tidy['task_status'].unique()}"
    )
    assert pd.api.types.is_numeric_dtype(df2_tidy['duration_seconds']), (
        "duration_seconds should be numeric (int or float)"
    )
    print("✅  Tidy check passed!")
    print(f"   Shape: {df2_tidy.shape}")
    print(f"   Columns: {df2_tidy.columns.tolist()}")
    print()
    print(df2_tidy.head(8).to_string(index=False))
except NameError:
    print("⚠️   df2_tidy is not defined — paste your code above and run it.")
except AssertionError as e:
    print(f"❌  Check failed: {e}")

---
## Exercise 3: Feature Reviews
### Violation: Multiple Observational Units in One Table

The research team also collected feature-by-feature ratings from a panel of UX
professionals. Each of the seven panelists reviewed all five FlowNote features and
left a short comment. The data was collected in a single spreadsheet.

In [ ]:
data3 = {
    'review_id': [f'RV{i:03d}' for i in range(1, 36)],
    'feature': [
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
        'Note Editor','Search','Sharing','Reminders','Organization',
    ],
    'rating': [
        4, 5, 4, 3, 5,
        3, 4, 3, 4, 4,
        5, 4, 4, 3, 5,
        4, 3, 5, 2, 4,
        3, 4, 3, 4, 3,
        4, 5, 4, 3, 4,
        3, 3, 4, 2, 4,
    ],
    'review_text': [
        'Clean interface, distraction-free writing works well.',
        'Full-text search is fast and accurate.',
        'Link sharing with permissions works smoothly.',
        'Time-based reminders work reliably.',
        'Nested folders help manage large libraries.',
        'Formatting toolbar is intuitive and responsive.',
        'Filters by tag are a great time saver.',
        'Team sharing UI needs clearer role controls.',
        'Snooze functionality needs more options.',
        'Tagging system is flexible and powerful.',
        'Markdown support is excellent for my workflow.',
        'Search results could rank recency higher.',
        'Shared notes sync quickly across devices.',
        'Reminder notifications blend in too much.',
        'Workspace concept could be clearer.',
        'Text rendering needs improvement on mobile.',
        'Boolean search operators would be helpful.',
        'Export to PDF option is very useful.',
        'Location reminders would be a nice addition.',
        'Bulk organization tools would save time.',
        'Font options limited but core editing is solid.',
        'Instant results as I type is excellent.',
        'Sharing via email could be more streamlined.',
        'Recurring reminder setup is straightforward.',
        'Color-coded notebooks aid visual scanning.',
        'Auto-save feature saved me multiple times.',
        'Search ignores note attachments unfortunately.',
        'Real-time collaboration is a key differentiator.',
        'Integration with calendar apps is missing.',
        'Drag-and-drop reordering works intuitively.',
        'Editor lags when notes get very long.',
        'Solid baseline, needs fuzzy matching support.',
        'Permission levels are confusing for new users.',
        'Reminder UI is clean but lacks customization.',
        'Archive feature prevents note clutter.',
    ],
    'reviewer_id': (
        ['R01']*5 + ['R02']*5 + ['R03']*5 + ['R04']*5 +
        ['R05']*5 + ['R06']*5 + ['R07']*5
    ),
    'reviewer_name': (
        ['Alex Chen']*5    + ['Priya Patel']*5  + ['Jordan Kim']*5 +
        ['Sam Okafor']*5   + ['Taylor Reyes']*5 + ['Morgan Liu']*5 +
        ['Casey Brown']*5
    ),
    'reviewer_role': (
        ['Senior UX Designer']*5 + ['Product Manager']*5 +
        ['Frontend Developer']*5 + ['UX Researcher']*5   +
        ['Content Strategist']*5 + ['UX Designer']*5     +
        ['Product Manager']*5
    ),
    'reviewer_company': (
        ['Figma']*5 + ['Spotify']*5 + ['Airbnb']*5 +
        ['Google']*5 + ['Adobe']*5  + ['Microsoft']*5 +
        ['Dropbox']*5
    ),
}

df3 = pd.DataFrame(data3)
df3.head(12)

### Step 1 — Explore the data

In [ ]:
print("Shape:", df3.shape)
print()
print("Columns:", df3.columns.tolist())
print()
print("Reviewer info (unique rows only):")
print(
    df3[['reviewer_id','reviewer_name','reviewer_role','reviewer_company']]
    .drop_duplicates()
    .to_string(index=False)
)

### Step 2 — Spot the violation

Look at the reviewer columns: `reviewer_id`, `reviewer_name`, `reviewer_role`,
`reviewer_company`.

> **Think about it:**
> - How many times does Alex Chen's name appear in the table?
> - Does Alex Chen's employer (Figma) tell you something about her *review* of
>   a specific feature — or something about *Alex Chen as a person*?
> - Are reviews and reviewer profiles the same *type* of observation?

**The violation:** This table mixes two observational units:
- **Reviews** (one observation per reviewer-feature pair)
- **Reviewer profiles** (one observation per reviewer)

Because reviewer profile data is the same for all five of a person's reviews, it
repeats five times per reviewer. This violates tidy rule #3.

**The fix:** Split `df3` into two separate DataFrames that link via `reviewer_id`.

**`reviews`** — one row per review (35 rows):

| review_id | reviewer_id | feature | rating | review_text |
|---|---|---|---|---|

**`reviewers`** — one row per reviewer (7 rows):

| reviewer_id | reviewer_name | reviewer_role | reviewer_company |
|---|---|---|---|

> **Why it matters:** If Alex Chen moves from Figma to a new company, the
> single-table format requires updating 5 rows. The normalized format requires
> updating 1.

---
### Step 3 — Prompt your AI agent

For this problem you need to communicate the idea of *splitting* a DataFrame into
two normalized tables, not just transforming columns. Your prompt should specify:

- What the current data looks like (columns, row count)
- What two DataFrames you want
- Which columns go in each
- Which column links them together

**Model prompt:**
> *"I have a pandas DataFrame `df3` with 35 rows and these columns: `review_id`,
> `feature`, `rating`, `review_text`, `reviewer_id`, `reviewer_name`,
> `reviewer_role`, `reviewer_company`. The reviewer columns repeat for every
> review that person wrote — there are 7 reviewers and each wrote 5 reviews.
> Normalize this into two DataFrames: `reviews` (columns: `review_id`,
> `reviewer_id`, `feature`, `rating`, `review_text`) and `reviewers` (columns:
> `reviewer_id`, `reviewer_name`, `reviewer_role`, `reviewer_company` — one row
> per unique reviewer, no duplicates). The tables should be joinable on
> `reviewer_id`."*

**Write your version below.**

### My Prompt (Exercise 3)

```
[your prompt here]
```

In [ ]:
# Paste the code your AI agent produced here.
# You should create two DataFrames: reviews and reviewers.

# ── YOUR CODE HERE ──────────────────────────────────────────────────────────

# reviews   = ...
# reviewers = ...

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────

try:
    assert len(reviews) == 35,   f"Expected 35 rows in reviews, got {len(reviews)}"
    assert len(reviewers) == 7,  f"Expected 7 rows in reviewers, got {len(reviewers)}"

    assert 'reviewer_id' in reviews.columns,   "reviews needs a reviewer_id column"
    assert 'reviewer_id' in reviewers.columns, "reviewers needs a reviewer_id column"

    redundant = {'reviewer_name','reviewer_role','reviewer_company'}
    overlap = redundant & set(reviews.columns)
    assert not overlap, (
        f"Reviewer profile columns should be in 'reviewers', not 'reviews': {overlap}"
    )

    assert reviewers['reviewer_id'].nunique() == 7, (
        "reviewers should have exactly 7 unique reviewer_ids"
    )

    # Confirm the two tables join cleanly
    merged = reviews.merge(reviewers, on='reviewer_id')
    assert len(merged) == 35, (
        f"Merge returned {len(merged)} rows — check that reviewer_id values match"
    )

    print("✅  Tidy check passed!")
    print(f"   reviews:   {reviews.shape}")
    print(f"   reviewers: {reviewers.shape}")
    print()
    print("── reviews (first 6 rows) ─────────────────────────────")
    print(reviews.head(6).to_string(index=False))
    print()
    print("── reviewers ──────────────────────────────────────────")
    print(reviewers.to_string(index=False))

except NameError as e:
    print(f"⚠️   {e} — paste your code above and run it.")
except AssertionError as e:
    print(f"❌  Check failed: {e}")

---
## Reflection

You've worked through three of Wickham's five common data-tidying problems.

### What you solved

| Exercise | Violation | Pandas fix |
|----------|-----------|------------|
| 1 — Satisfaction Ratings | Column headers were values (week numbers) | `pd.melt()` — wide → long |
| 2 — Usability Test Log | Multiple variables packed into one column | `str.split()` — separate combined fields |
| 3 — Feature Reviews | Two observational units in one table | Subset columns → two normalized DataFrames |

### Discussion questions

1. **Prompting quality.** Look back at the prompts you wrote. What made a prompt
   work well? What did you have to revise when the agent got it wrong?

2. **Recognizing the problem.** Which dataset was hardest to identify as "not
   tidy"? What gave it away once you found it?

3. **Sequencing.** Real-world data often combines all three problems at once. If
   you encountered a dataset with multiple issues, how would you order the fixes?

4. **The two remaining Wickham violations** not covered here are:
   - *Variables in both rows and columns* (e.g., a pivot table where some rows
     are summary labels)
   - *One observational unit spread across multiple files* (e.g., monthly exports
     that need to be concatenated)

   Can you sketch what those would look like using the FlowNote scenario?

### Going further

Once your data is tidy, analysis becomes much simpler to write:

```python
# Satisfaction trend by week (needs tidy df1_tidy)
df1_tidy.groupby('week')['satisfaction_score'].mean().plot(marker='o')

# Completion rate by skill level (needs tidy df2_tidy)
df2_tidy.groupby('skill_level')['task_status'].apply(
    lambda s: (s == 'completed').mean()
)

# Average rating per feature with reviewer info (needs normalized df3)
reviews.merge(reviewers, on='reviewer_id').groupby('feature')['rating'].mean()
```

Each of these would be much harder — or impossible — to write against the
original messy formats.